In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="text-align:right;padding:8px 0;">
<button id="toggle-code-btn" style="
  padding:8px 18px;font-size:14px;cursor:pointer;
  background:#4a90d9;color:white;border:none;border-radius:5px;
  box-shadow:0 2px 4px rgba(0,0,0,0.2);
" onclick="
  var cells = document.querySelectorAll('.jp-Cell-inputWrapper');
  var btn = document.getElementById('toggle-code-btn');
  var show = btn.dataset.show !== 'true';
  btn.dataset.show = show;
  btn.textContent = show ? '隐藏代码 [Hide]' : '显示代码 [Show]';
  cells.forEach(function(c){ c.style.display = show ? '' : 'none'; });
">显示代码 [Show]</button>
</div>
'''))


# 第5章 系统软件 - 5.1 操作系统 (Operating System)

**Cambridge International AS & A Level Computer Science (9618)**

---

在这一节中，我们将深入探索**操作系统 (Operating System, OS)** —— 计算机中最重要的系统软件。
我们将回答一个根本问题：**为什么我们需要操作系统？没有它会怎样？**

> **学习目标：**
> - 理解操作系统的定义和核心功能
> - 掌握内存管理、文件管理、安全管理、硬件管理和进程管理
> - 了解实用工具软件和程序库
> - 能够用Python模拟和可视化操作系统概念

## 1. 为什么我们需要操作系统？

### 1.1 没有操作系统的世界

想象一下早期的计算机（1940-1950年代）：

- 程序员必须**直接操作硬件** —— 用物理开关、打孔卡片来输入每一条指令
- 每次只能运行**一个程序**，运行完毕后必须手动加载下一个
- 如果程序想把数据存到磁盘，程序员必须知道磁盘的**每个物理扇区地址**
- 换一台打印机？必须**重写所有打印相关的代码**

这就像没有服务员的餐厅 —— 你需要自己去厨房拿食材、自己做饭、自己洗碗。

### 1.2 操作系统的诞生

**操作系统就是这个"服务员"** —— 它在你和厨房（硬件）之间充当中介：

| 没有OS | 有OS |
|--------|------|
| 程序员直接控制硬件 | OS代理硬件操作 |
| 一次只能运行一个程序 | 可以同时"运行"多个程序 |
| 必须知道硬件细节 | OS提供统一接口 |
| 换硬件需要改程序 | OS通过驱动程序适配硬件 |

> **定义：Operating System (操作系统)** 是一组管理计算机硬件资源、为应用程序提供服务、
> 并为用户提供交互界面的系统软件。

### 1.3 操作系统的分层结构

操作系统就像一个**洋葱的层次结构** —— 最内层是硬件，最外层是用户：

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 WenQuanYi Zen Hei 字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('WenQuanYi Zen Hei')
if 'WenQuanYi Zen Hei' in test_font or 'wqy' in test_font.lower():
    print('✅ 中文字体 WenQuanYi Zen Hei 已加载')
else:
    print(f'⚠️ WenQuanYi Zen Hei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 WenQuanYi Zen Hei 字体文件')

import matplotlib.patches as patches
import numpy as np

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Draw concentric rounded rectangles (layers)
layers = [
    {'name': 'User (用户)', 'color': '#E8F5E9', 'y': 0, 'h': 8, 'w': 10},
    {'name': 'Application Software\n(应用软件: Word, Chrome, 游戏...)', 'color': '#C8E6C9', 'y': 0.8, 'h': 6.4, 'w': 8.4},
    {'name': 'Operating System (操作系统)\nMemory | File | Process | Security | I/O', 'color': '#81C784', 'y': 1.6, 'h': 4.8, 'w': 6.8},
    {'name': 'Device Drivers (设备驱动)', 'color': '#4CAF50', 'y': 2.4, 'h': 3.2, 'w': 5.2},
    {'name': 'Hardware (硬件)\nCPU | RAM | Disk | I/O', 'color': '#2E7D32', 'y': 3.0, 'h': 2.0, 'w': 3.6},
]

for layer in layers:
    x = (10 - layer['w']) / 2
    rect = patches.FancyBboxPatch(
        (x, layer['y']), layer['w'], layer['h'],
        boxstyle='round,pad=0.2', facecolor=layer['color'],
        edgecolor='black', linewidth=2
    )
    ax.add_patch(rect)
    ax.text(5, layer['y'] + layer['h'] - 0.5, layer['name'],
            ha='center', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.5, 8.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Operating System Layer Model (操作系统分层模型)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 2. 内存管理 (Memory Management)

### 2.1 为什么需要内存管理？

把RAM想象成一个**大办公桌**：

- 办公桌大小是固定的（比如8GB RAM）
- 每个运行中的程序都需要占一块桌面空间
- 操作系统本身也需要空间
- 如果桌面不够用怎么办？

**内存管理的核心任务：**

1. **分配 (Allocation)**：当程序启动时，为它分配一块内存空间
2. **跟踪 (Tracking)**：记住哪些内存正在使用，哪些是空闲的
3. **保护 (Protection)**：确保程序A不能读写程序B的内存空间
4. **回收 (Deallocation)**：当程序结束时，释放它占用的内存
5. **虚拟内存 (Virtual Memory)**：当RAM不够时，用硬盘空间临时充当RAM

> **真实世界类比：** 内存管理就像酒店前台 —— 给客人分配房间（分配内存），记录哪些房间有人住（跟踪），
> 确保客人不能进别人房间（保护），客人退房后清理房间（回收）。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

def draw_memory(ax, title, blocks):
    """Draw a memory block diagram."""
    ax.set_xlim(0, 4)
    ax.set_ylim(0, 10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')
    y = 0
    for name, size, color in blocks:
        rect = patches.Rectangle((0.5, y), 3, size, facecolor=color,
                                  edgecolor='black', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(2, y + size/2, f'{name}\n({size}GB)',
                ha='center', va='center', fontsize=10, fontweight='bold')
        y += size

# Stage 1: Initial state
draw_memory(axes[0], 'Stage 1: OS loaded (OS已加载)',
            [('OS', 2, '#FF8A80'), ('Free (空闲)', 6, '#E0E0E0')])

# Stage 2: Programs loaded
draw_memory(axes[1], 'Stage 2: Programs running (程序运行中)',
            [('OS', 2, '#FF8A80'), ('Chrome', 2, '#82B1FF'),
             ('Game (游戏)', 3, '#B9F6CA'), ('Free', 1, '#E0E0E0')])

# Stage 3: Program closed, fragmentation
draw_memory(axes[2], 'Stage 3: Chrome closed (Chrome关闭)',
            [('OS', 2, '#FF8A80'), ('Free', 2, '#E0E0E0'),
             ('Game (游戏)', 3, '#B9F6CA'), ('Free', 1, '#E0E0E0')])

fig.suptitle('Memory Allocation Over Time (内存分配过程)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. 文件管理 (File Management)

### 3.1 文件系统 (File System) 是什么？

硬盘就像一个**巨大的仓库**，里面有成千上万个货架（扇区）。
如果没有管理系统，你怎么知道你的照片存在哪个货架上？

**文件系统 (File System)** 就是这个仓库的管理手册 —— 它定义了：
- 数据在磁盘上如何组织
- 文件名和物理位置的对应关系
- 文件的元数据（大小、创建日期、权限等）

### 3.2 常见文件系统对比

| 文件系统 | 全称 | 常用于 | 最大文件大小 | 特点 |
|---------|------|--------|------------|------|
| **FAT32** | File Allocation Table 32 | U盘、SD卡 | 4GB | 兼容性最好，但限制多 |
| **NTFS** | New Technology File System | Windows | 16EB | 支持权限、加密、日志 |
| **ext4** | Fourth Extended File System | Linux | 16TB | Linux标准，高效可靠 |
| **APFS** | Apple File System | macOS/iOS | 8EB | 苹果设备专用，优化SSD |

### 3.3 目录结构 (Directory Structure)

文件系统采用**树形层次结构 (Tree Hierarchy)** —— 就像一棵倒着长的树：
- **根目录 (Root Directory)**：树的根部（Windows是 `C:\`，Linux是 `/`）
- **子目录 (Subdirectory)**：树的分支
- **文件 (File)**：树的叶子

### 3.4 文件的基本操作

| 操作 | 英文 | 说明 |
|------|------|------|
| 创建 | Create | 在指定位置创建新文件 |
| 读取 | Read | 从文件中获取数据 |
| 写入 | Write | 向文件中存入数据 |
| 删除 | Delete | 移除文件（通常只是删除索引条目） |
| 重命名 | Rename | 更改文件名称 |
| 复制 | Copy | 在新位置创建文件副本 |

### 3.5 文件权限 (File Permissions)

操作系统控制**谁可以对哪些文件做什么操作**。

在Linux/Unix系统中，每个文件有三组权限：

| 权限 | 符号 | 数字 | 含义 |
|------|------|------|------|
| Read（读） | r | 4 | 可以查看文件内容 |
| Write（写） | w | 2 | 可以修改文件内容 |
| Execute（执行） | x | 1 | 可以运行此文件 |

三组分别对应：**Owner（所有者）**、**Group（组）**、**Others（其他人）**

例如 `chmod 754` 意味着：
- Owner: 7 = 4+2+1 = rwx（读写执行）
- Group: 5 = 4+0+1 = r-x（读和执行）
- Others: 4 = 4+0+0 = r--（只读）

### 3.6 互动实验：模拟一个简单的文件系统

In [ ]:
class SimpleFileSystem:
    """A simple simulation of a file system with directory tree."""
    
    def __init__(self):
        # The file system is a nested dictionary
        # Directories map to dicts, files map to strings (content)
        self.root = {
            'type': 'dir',
            'children': {},
            'name': '/'
        }
        self.cwd = self.root  # current working directory
        self.cwd_path = '/'
    
    def mkdir(self, name):
        """Create a new directory."""
        if name in self.cwd['children']:
            print(f'Error: "{name}" already exists!')
            return
        self.cwd['children'][name] = {
            'type': 'dir', 'children': {}, 'name': name
        }
        print(f'Directory created: {self.cwd_path}{name}/')
    
    def create_file(self, name, content=''):
        """Create a new file with optional content."""
        self.cwd['children'][name] = {
            'type': 'file', 'content': content, 'name': name,
            'size': len(content)
        }
        print(f'File created: {self.cwd_path}{name} ({len(content)} bytes)')
    
    def ls(self):
        """List contents of current directory."""
        print(f'Contents of {self.cwd_path}:')
        if not self.cwd['children']:
            print('  (empty)')
            return
        for name, item in self.cwd['children'].items():
            if item['type'] == 'dir':
                print(f'  [DIR]  {name}/')
            else:
                print(f'  [FILE] {name} ({item["size"]} bytes)')
    
    def cd(self, name):
        """Change to a subdirectory."""
        if name == '..':
            # Simplified: go back to root
            self.cwd = self.root
            self.cwd_path = '/'
            print(f'Changed to {self.cwd_path}')
            return
        if name not in self.cwd['children']:
            print(f'Error: "{name}" not found!')
            return
        if self.cwd['children'][name]['type'] != 'dir':
            print(f'Error: "{name}" is not a directory!')
            return
        self.cwd = self.cwd['children'][name]
        self.cwd_path += f'{name}/'
        print(f'Changed to {self.cwd_path}')
    
    def read_file(self, name):
        """Read file content."""
        if name not in self.cwd['children']:
            print(f'Error: "{name}" not found!')
            return
        item = self.cwd['children'][name]
        if item['type'] != 'file':
            print(f'Error: "{name}" is a directory, not a file!')
            return
        print(f'--- Content of {name} ---')
        print(item['content'])
    
    def delete(self, name):
        """Delete a file or empty directory."""
        if name not in self.cwd['children']:
            print(f'Error: "{name}" not found!')
            return
        del self.cwd['children'][name]
        print(f'Deleted: {name}')

# Demo
fs = SimpleFileSystem()
print('=== Simulating a File System ===')
print()

fs.mkdir('Documents')
fs.mkdir('Pictures')
fs.mkdir('Music')
print()

fs.ls()
print()

fs.cd('Documents')
fs.create_file('homework.txt', 'This is my CS homework about operating systems.')
fs.create_file('notes.txt', 'Memory management, file management, security...')
fs.ls()
print()

fs.read_file('homework.txt')
print()

fs.delete('notes.txt')
fs.ls()

---
## 4. 安全管理 (Security Management)

### 4.1 为什么操作系统需要安全管理？

想象一栋办公楼：
- 不是每个人都能进入每个房间
- 访客需要在前台**登记身份**
- 不同员工有**不同的门禁权限**
- 安保系统会**记录所有进出**

操作系统的安全管理做的就是同样的事情：

### 4.2 用户账户与认证 (User Accounts & Authentication)

- **用户名 + 密码**：最基本的认证方式
- **多因素认证 (MFA)**：密码 + 手机验证码 / 指纹
- **用户组 (User Groups)**：将用户分组，方便批量管理权限
  - 例如：管理员组 (Administrators)、普通用户组 (Users)、访客组 (Guests)

### 4.3 访问控制级别 (Access Control Levels)

| 级别 | 英文 | 权限范围 |
|------|------|----------|
| 超级管理员 | Root / Administrator | 完全控制，可修改系统任何部分 |
| 普通用户 | Standard User | 可运行程序，只能修改自己的文件 |
| 访客 | Guest | 最小权限，不能安装软件或更改设置 |

### 4.4 操作系统如何防止未授权访问？

1. **登录验证**：进入系统前必须通过身份验证
2. **文件权限**：控制谁可以读/写/执行哪些文件
3. **内存保护**：每个进程只能访问自己的内存空间
4. **防火墙 (Firewall)**：监控和过滤网络流量
5. **审计日志 (Audit Log)**：记录所有安全相关事件

In [ ]:
# Simulating a simple login system
import hashlib

class SimpleAuthSystem:
    """A simplified authentication system demonstration."""
    
    def __init__(self):
        self.users = {}
        self.access_levels = {
            'admin': ['read', 'write', 'execute', 'install', 'manage_users'],
            'user': ['read', 'write', 'execute'],
            'guest': ['read']
        }
    
    def register(self, username, password, role='user'):
        # In real OS, passwords are hashed, never stored in plaintext!
        hashed = hashlib.sha256(password.encode()).hexdigest()
        self.users[username] = {'password_hash': hashed, 'role': role}
        print(f'User "{username}" registered as {role}')
        print(f'  Password hash: {hashed[:20]}...  (never store plaintext!)')
    
    def login(self, username, password):
        if username not in self.users:
            print(f'Login FAILED: User "{username}" not found')
            return False
        hashed = hashlib.sha256(password.encode()).hexdigest()
        if hashed != self.users[username]['password_hash']:
            print(f'Login FAILED: Wrong password for "{username}"')
            return False
        role = self.users[username]['role']
        perms = self.access_levels[role]
        print(f'Login SUCCESS: Welcome, {username}!')
        print(f'  Role: {role}')
        print(f'  Permissions: {perms}')
        return True

# Demo
auth = SimpleAuthSystem()
print('=== Simple Authentication System ===')
print()

auth.register('admin_zhang', 'Secure@123', 'admin')
auth.register('student_li', 'MyPassword', 'user')
auth.register('visitor', 'guest123', 'guest')
print()

auth.login('admin_zhang', 'Secure@123')
print()
auth.login('student_li', 'WrongPassword')  # Wrong password
print()
auth.login('student_li', 'MyPassword')  # Correct password

---
## 5. 硬件/输入输出管理 (Hardware / I/O Management)

### 5.1 设备驱动程序 (Device Drivers)

**问题：** 世界上有成千上万种不同的打印机、显卡、键盘...
操作系统不可能知道每一种设备的具体操作方式。

**解决方案：设备驱动程序 (Device Driver)**

设备驱动就像**翻译官**：
- 操作系统说："请打印这个文档"（统一的高级命令）
- 驱动程序翻译成：该打印机能理解的具体指令序列

```
应用程序 ---> 操作系统 ---> 设备驱动 ---> 硬件设备
  "打印"       "Print()"    具体的电信号    打印机工作
```

### 5.2 即插即用 (Plug and Play, PnP)

早期：安装新设备需要手动配置中断号、I/O端口、安装驱动 —— 非常痛苦！

**即插即用 (Plug and Play)**：
1. 插入新设备
2. 操作系统**自动检测**设备
3. 操作系统**自动查找并安装**驱动程序
4. 设备立即可用！

> 就像住酒店 —— 你只需要走进房间，灯、空调、电视都已经准备好了，不需要你自己接线。

### 5.3 假脱机 (Spooling)

**问题：** 打印机很慢，如果10个人同时要打印，怎么办？

**Spooling** = Simultaneous Peripheral Operations On-Line

工作原理：
1. 每个打印请求不是直接发给打印机
2. 而是先存入一个**打印队列 (Print Queue)** —— 一个磁盘上的缓冲区
3. 打印机按照**先进先出 (FIFO)** 的顺序，一个一个地处理
4. 用户的程序立即"完成"打印操作，不需要等待实际打印

> 就像在银行取号 —— 你取了号就可以坐下等，不需要站在柜台前排队。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(14, 5))

# Users
users_x = [0.5, 0.5, 0.5]
users_y = [3.5, 2.0, 0.5]
labels = ['User A: Print report', 'User B: Print essay', 'User C: Print slides']

for x, y, label in zip(users_x, users_y, labels):
    rect = patches.FancyBboxPatch((x, y), 3, 0.8, boxstyle='round,pad=0.1',
                                   facecolor='#BBDEFB', edgecolor='black')
    ax.add_patch(rect)
    ax.text(x + 1.5, y + 0.4, label, ha='center', va='center', fontsize=9)

# Arrows to spool
for y in [3.9, 2.4, 0.9]:
    ax.annotate('', xy=(4.5, 2.4), xytext=(3.5, y),
                arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))

# Spool (queue)
spool_rect = patches.FancyBboxPatch((4.5, 1.2), 3.5, 2.4,
                                     boxstyle='round,pad=0.2',
                                     facecolor='#FFF9C4', edgecolor='black', lw=2)
ax.add_patch(spool_rect)
ax.text(6.25, 3.2, 'Print Spool (Queue)', ha='center', va='center',
        fontsize=11, fontweight='bold')
# Queue items
for i, (task, color) in enumerate([
    ('Job 1: Report', '#EF9A9A'),
    ('Job 2: Essay', '#A5D6A7'),
    ('Job 3: Slides', '#90CAF9')
]):
    r = patches.Rectangle((4.8, 2.5 - i * 0.5), 2.9, 0.4,
                           facecolor=color, edgecolor='black')
    ax.add_patch(r)
    ax.text(6.25, 2.7 - i * 0.5, task, ha='center', va='center', fontsize=9)

# Arrow to printer
ax.annotate('', xy=(9, 2.4), xytext=(8, 2.4),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))

# Printer
printer = patches.FancyBboxPatch((9, 1.6), 2.5, 1.6,
                                  boxstyle='round,pad=0.2',
                                  facecolor='#E0E0E0', edgecolor='black', lw=2)
ax.add_patch(printer)
ax.text(10.25, 2.4, 'Printer\n(打印机)\nProcessing\nJob 1...', ha='center', va='center',
        fontsize=10, fontweight='bold')

ax.set_xlim(0, 12)
ax.set_ylim(0, 4.8)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Spooling: Print Queue System (假脱机：打印队列系统)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. 进程管理 (Process Management)

### 6.1 什么是进程 (Process)？

**程序 (Program)** vs **进程 (Process)**：

| 程序 (Program) | 进程 (Process) |
|----------------|----------------|
| 存储在磁盘上的代码 | 正在内存中运行的程序 |
| 静态的，不占CPU | 动态的，使用CPU和内存 |
| 一个程序 | 可以产生多个进程 |

> **类比：** 程序就像一本食谱（静静地放在书架上），进程就是你按照食谱**正在做菜**的过程。
> 同一本食谱可以被多个厨师同时使用 = 同一个程序可以有多个进程。

### 6.2 多任务 (Multitasking) 的奥秘

**核心问题：** 一个CPU在任意时刻只能执行一条指令。那么它是如何"同时"运行多个程序的？

**答案：时间片轮转 (Time Slicing / Round Robin)**

1. OS把CPU时间切成很小的片段（通常是几毫秒）
2. 每个进程轮流获得一个时间片
3. 时间片用完后，OS保存当前进程的状态，切换到下一个进程
4. 切换速度非常快（每秒数千次），人类感觉不到切换

这就是为什么你**感觉**电脑在同时播放音乐、显示网页、运行游戏 —— 实际上CPU在这些任务之间快速切换！

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, ax = plt.subplots(figsize=(16, 5))

# Define processes and colors
processes = {
    'Chrome': '#42A5F5',
    'Music Player': '#66BB6A',
    'Word': '#FFA726',
    'Antivirus': '#EF5350'
}

# Simulate time slices (round robin)
proc_list = list(processes.keys())
n_slices = 16
schedule = [proc_list[i % len(proc_list)] for i in range(n_slices)]

# Draw timeline
for i, proc in enumerate(schedule):
    rect = patches.Rectangle((i * 0.9 + 0.5, 1.5), 0.85, 1.5,
                              facecolor=processes[proc],
                              edgecolor='black', linewidth=1)
    ax.add_patch(rect)
    ax.text(i * 0.9 + 0.925, 2.25, proc.split()[0],
            ha='center', va='center', fontsize=7, rotation=45)
    ax.text(i * 0.9 + 0.925, 1.2, f'{i}ms',
            ha='center', va='center', fontsize=7)

# Legend
legend_y = 4.0
for i, (name, color) in enumerate(processes.items()):
    rect = patches.Rectangle((1 + i * 3.8, legend_y), 0.5, 0.4,
                              facecolor=color, edgecolor='black')
    ax.add_patch(rect)
    ax.text(1.7 + i * 3.8, legend_y + 0.2, name,
            va='center', fontsize=10)

# Arrow for time direction
ax.annotate('', xy=(15.5, 1.0), xytext=(0.5, 1.0),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(8, 0.6, 'Time (时间) --->', ha='center', fontsize=12)

ax.set_xlim(0, 16.5)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('CPU Time Slicing / Round Robin (CPU时间片轮转)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. 实用工具软件 (Utility Software)

**实用工具软件 (Utility Software)** 是操作系统附带的（或可以额外安装的）小程序，
用于**维护和优化**计算机系统。

就像汽车需要定期保养（换机油、检查轮胎、清洗），计算机也需要定期维护：

| 工具 | 英文 | 功能 | 为什么需要？ |
|------|------|------|-------------|
| 磁盘格式化工具 | Disk Formatter | 初始化磁盘，创建文件系统 | 新磁盘必须格式化后才能存储文件 |
| 病毒扫描程序 | Virus Checker | 检测和清除恶意软件 | 保护系统免受恶意程序攻击 |
| 磁盘碎片整理 | Defragmenter | 重新排列磁盘上的数据碎片 | 提高机械硬盘的读写速度 |
| 磁盘修复工具 | Disk Repair | 检查和修复磁盘错误 | 修复损坏的文件系统结构 |
| 文件压缩工具 | File Compression | 减小文件大小 | 节省存储空间，加快传输速度 |
| 备份工具 | Backup | 创建数据副本 | 防止数据丢失（硬盘故障、误删除等） |

### 7.1 深入理解：磁盘碎片整理 (Defragmentation)

**为什么会产生碎片？**

当你不断创建、修改、删除文件时，文件的各个部分可能散布在磁盘的不同位置。
这就像一本书的页面被撕开放在图书馆的不同书架上 —— 要读完整本书，你需要跑来跑去！

**碎片整理做什么？**

把同一个文件的所有碎片重新排列到连续的位置，让磁头可以一次顺序读取。

> **注意：** SSD（固态硬盘）**不需要**碎片整理！因为SSD没有物理磁头，随机读取和顺序读取速度几乎相同。
> 对SSD进行碎片整理反而会**缩短寿命**（因为SSD有写入次数限制）。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Color mapping for files
file_colors = {
    'A': '#EF5350',  # Red
    'B': '#42A5F5',  # Blue
    'C': '#66BB6A',  # Green
    'D': '#FFA726',  # Orange
    ' ': '#EEEEEE',  # Empty
}

# Before defragmentation - fragmented layout
before = ['A','B','A',' ','C','B','A','D',' ','C','B','D','A','C',' ','D',
          ' ','B','C','D','A',' ',' ','A']

# After defragmentation - contiguous layout
after = ['A','A','A','A','A','A','B','B','B','B','C','C','C','C',
         'D','D','D','D',' ',' ',' ',' ',' ',' ']

for ax, data, title in [
    (axes[0], before, 'BEFORE Defragmentation (碎片整理前) - Files scattered!'),
    (axes[1], after, 'AFTER Defragmentation (碎片整理后) - Files contiguous!')
]:
    for i, block in enumerate(data):
        rect = patches.Rectangle((i * 0.6, 0), 0.55, 1,
                                  facecolor=file_colors[block],
                                  edgecolor='black', linewidth=1)
        ax.add_patch(rect)
        if block != ' ':
            ax.text(i * 0.6 + 0.275, 0.5, block,
                    ha='center', va='center', fontsize=12, fontweight='bold')
    ax.set_xlim(-0.1, 14.5)
    ax.set_ylim(-0.3, 1.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

# Legend
legend_text = '  '.join([f'File {k} = {k}' for k in ['A','B','C','D']] + ['Empty = [ ]'])
fig.text(0.5, 0.01, 'Legend: File A (Red) | File B (Blue) | File C (Green) | File D (Orange) | Empty (Gray)',
         ha='center', fontsize=11)

fig.suptitle('Disk Defragmentation Visualization (磁盘碎片整理可视化)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(top=0.88, bottom=0.08)
plt.show()

### 7.2 互动实验：文件压缩原理

In [ ]:
import zlib

# Demonstrate compression
print('=== File Compression Demo (文件压缩演示) ===')
print()

# Text with lots of repetition (compresses well)
text1 = 'AAAAAABBBBBBCCCCCCDDDDDDEEEEEE' * 100
compressed1 = zlib.compress(text1.encode())
ratio1 = len(compressed1) / len(text1) * 100

print(f'Text 1 (very repetitive / 高重复):')
print(f'  Original size:    {len(text1)} bytes')
print(f'  Compressed size:  {len(compressed1)} bytes')
print(f'  Compression ratio: {ratio1:.1f}% (smaller = better)')
print()

# Text with less repetition
import random
random.seed(42)
text2 = ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789', k=3000))
compressed2 = zlib.compress(text2.encode())
ratio2 = len(compressed2) / len(text2) * 100

print(f'Text 2 (random / 随机):')
print(f'  Original size:    {len(text2)} bytes')
print(f'  Compressed size:  {len(compressed2)} bytes')
print(f'  Compression ratio: {ratio2:.1f}% (smaller = better)')
print()

print('Conclusion (结论):')
print('  Repetitive data compresses MUCH better than random data!')
print('  压缩算法通过找到重复模式来减小文件大小。')
print('  重复越多 -> 压缩效果越好！')

---
## 8. 程序库 (Program Libraries)

### 8.1 什么是程序库？

**程序库 (Program Library)** 是一组**预先编写好的、可重复使用的代码模块**。

**为什么需要程序库？**

想象你在建房子：
- 你**不需要**自己烧砖、炼钢、造玻璃
- 你直接从工厂**购买**标准化的砖块、钢梁、玻璃窗
- 然后**组装**成你想要的房子

程序库就是编程世界里的"标准化零件"：

| 好处 | 说明 |
|------|------|
| **节省时间** | 不需要重新发明轮子 |
| **减少错误** | 库代码经过大量测试和验证 |
| **标准化** | 多个程序使用同一套库，行为一致 |
| **专业性** | 库通常由专家编写，质量更高 |

### 8.2 程序库的例子

| 语言 | 库名 | 用途 |
|------|------|------|
| Python | `math` | 数学运算 |
| Python | `matplotlib` | 绑图和可视化 |
| Python | `os` | 操作系统交互 |
| C | `stdio.h` | 标准输入输出 |
| Java | `java.util` | 工具类集合 |

### 8.3 两种链接方式

- **静态链接 (Static Linking)**：库代码在编译时被复制到可执行文件中
  - 优点：程序独立，不依赖外部文件
  - 缺点：文件更大，更新库需要重新编译

- **动态链接 (Dynamic Linking)**：程序运行时才加载库（如 Windows 的 .dll 文件）
  - 优点：文件更小，多个程序共享同一个库文件
  - 缺点：如果库文件缺失或版本不对，程序会崩溃（"DLL Hell"）

In [ ]:
# Demonstrating the power of libraries
print('=== The Power of Program Libraries ===')
print()

# Without a library - manual calculation
print('--- Without math library (手动计算平方根) ---')
# Newton's method for square root
def my_sqrt(n, precision=0.0001):
    guess = n / 2.0
    iterations = 0
    while abs(guess * guess - n) > precision:
        guess = (guess + n / guess) / 2.0
        iterations += 1
    return guess, iterations

result, iters = my_sqrt(144)
print(f'  sqrt(144) = {result} (took {iters} iterations to calculate)')
print()

# With a library - one line!
import math
print('--- With math library (使用库) ---')
print(f'  math.sqrt(144) = {math.sqrt(144)}')
print(f'  math.pi = {math.pi}')
print(f'  math.factorial(10) = {math.factorial(10)}')
print()

# Show how many functions are in the math library
math_funcs = [x for x in dir(math) if not x.startswith('_')]
print(f'The math library provides {len(math_funcs)} ready-to-use functions!')
print(f'Examples: {math_funcs[:10]}...')
print()
print('Without libraries, you would have to write ALL of these yourself!')
print('库为我们节省了大量时间，让我们专注于解决真正的问题。')

---
## 9. 本节总结

```
Operating System (操作系统)
|
+-- Memory Management (内存管理)
|   +-- Allocation / Deallocation (分配/回收)
|   +-- Memory Protection (内存保护)
|   +-- Virtual Memory (虚拟内存)
|
+-- File Management (文件管理)
|   +-- File Systems: FAT32, NTFS, ext4
|   +-- Directory Structure (目录结构)
|   +-- File Operations & Permissions (文件操作与权限)
|
+-- Security Management (安全管理)
|   +-- Authentication (身份验证)
|   +-- Access Control (访问控制)
|   +-- Audit Logging (审计日志)
|
+-- Hardware/IO Management (硬件/IO管理)
|   +-- Device Drivers (设备驱动)
|   +-- Plug and Play (即插即用)
|   +-- Spooling (假脱机)
|
+-- Process Management (进程管理)
|   +-- Process vs Program (进程 vs 程序)
|   +-- Time Slicing / Multitasking (时间片/多任务)
|
+-- Utility Software (实用工具软件)
|   +-- Formatter, Virus Checker, Defragmenter...
|
+-- Program Libraries (程序库)
    +-- Static vs Dynamic Linking
```

---
## 10. 练习题 (Practice Exercises)

### 选择题

**Q1.** 以下哪个不是操作系统的核心功能？
- A. 内存管理
- B. 编写应用程序
- C. 文件管理
- D. 进程管理

**Q2.** 关于Spooling，以下哪个描述是正确的？
- A. 它让打印机运行更快
- B. 它将打印任务存入队列，让用户不必等待
- C. 它是一种内存管理技术
- D. 它只能用于键盘输入

**Q3.** 为什么SSD不需要磁盘碎片整理？
- A. SSD不会产生碎片
- B. SSD的随机读取和顺序读取速度几乎相同
- C. SSD容量太大，碎片无所谓
- D. SSD是只读设备

### 简答题

**Q4.** 解释Program和Process的区别。用一个现实世界的类比来说明。

**Q5.** 描述操作系统如何使用Time Slicing实现多任务处理。为什么用户感觉不到切换？

**Q6.** 解释Device Driver的作用。为什么操作系统不直接与所有硬件通信？

**Q7.** 比较Static Linking和Dynamic Linking的优缺点。

In [ ]:
# Practice Answers (练习答案)
answers = {
    'Q1': 'B - 编写应用程序不是OS的功能，那是程序员的工作。OS管理硬件、内存、文件、进程和安全。',
    'Q2': 'B - Spooling将打印任务存入磁盘队列，用户程序可以立即继续工作，不必等待慢速打印机。',
    'Q3': 'B - SSD使用闪存芯片，没有机械磁头，随机读取和顺序读取速度几乎相同，所以碎片不会影响性能。'
}

print('=== Answers to Multiple Choice (选择题答案) ===')
for q, a in answers.items():
    print(f'{q}: {a}')
    print()